In [1]:
import os
import csv
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

###############################################################################
#                             CONFIGURATION                                   #
###############################################################################
CSV_PATH = "one_video_per_class.csv"  # CSV listing your 20 .npy files
# We assume ALL lines in this CSV have split="train" now.

# If each video has exactly 120 frames, shape (120, H, W, 3)
FRAMES_PER_VIDEO = 120  

# Downsample frames to (64 x 64) to keep input size manageable
DOWNSAMPLE_TO = (64, 64)

# LSTM Input Dimension = 64*64*3 = 12288
INPUT_DIM = DOWNSAMPLE_TO[0] * DOWNSAMPLE_TO[1] * 3

# Model Hyperparameters
HIDDEN_DIM = 256    # LSTM hidden size
NUM_EPOCHS = 5      # Number of epochs
BATCH_SIZE = 1      # With so few samples, batch size of 1 is fine
LEARNING_RATE = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

###############################################################################
#                            DATASET DEFINITION                               #
###############################################################################
class RawVideoDataset(Dataset):
    """
    Loads a .npy file (one per sample), each is (T, H, W, 3).
    Downsamples each frame to (64,64), flattens to 1D, stacks into (T, input_dim).
    Returns a (T, input_dim) float32 tensor and an integer label.
    """
    def __init__(self, csv_file, split="train", resize_shape=(64, 64)):
        super().__init__()
        self.resize_shape = resize_shape
        self.samples = []
        
        # 1. Read CSV, pick rows matching 'split'
        with open(csv_file, "r") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
        
        for row in rows:
            if row["split"] == split:
                # row["filepath"] => e.g. "one_video_per_class/answer-1.npy"
                # row["label"]    => e.g. "answer"
                self.samples.append((row["filepath"], row["label"]))
        
        # 2. Build label -> index mapping
        unique_labels = sorted(list(set([s[1] for s in self.samples])))
        self.label_to_idx = {lbl: i for i, lbl in enumerate(unique_labels)}
        
        # 3. Convert labels to indices
        self.samples = [(fp, self.label_to_idx[lab]) for (fp, lab) in self.samples]

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        filepath, label_idx = self.samples[idx]
        
        # Load frames: shape (120, H, W, 3)
        frames = np.load(filepath)  # presumably (120, 256, 256, 3)
        
        processed_frames = []
        for frame in frames:
            # Downsample each frame to (64,64)
            small_frame = cv2.resize(frame, self.resize_shape)  # shape (64, 64, 3)
            # Flatten to 1D
            flat_frame = small_frame.reshape(-1)  # shape (64*64*3,)
            processed_frames.append(flat_frame)
        
        # Stack into shape (120, input_dim)
        processed_frames = np.array(processed_frames, dtype=np.float32)
        
        # Convert to torch tensor
        frames_tensor = torch.from_numpy(processed_frames)  # shape (120, input_dim)
        
        return frames_tensor, label_idx

###############################################################################
#                            LSTM MODEL DEFINITION                            #
###############################################################################
class VideoLSTM(nn.Module):
    """
    Simple LSTM that reads (batch, 120, input_dim) and outputs class logits.
    """
    def __init__(self, input_dim, hidden_dim, num_classes):
        super(VideoLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_dim,
                            hidden_size=hidden_dim,
                            batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        """
        x: shape (batch_size, 120, input_dim)
        """
        out, (h_n, c_n) = self.lstm(x)  # out: (batch_size, 120, hidden_dim)
        # Take the last timestep's output
        last_out = out[:, -1, :]       # (batch_size, hidden_dim)
        logits = self.fc(last_out)     # (batch_size, num_classes)
        return logits

###############################################################################
#                             TRAIN-ONLY LOOP                                #
###############################################################################
def train_only_lstm():
    # 1. Create dataset (train only)
    train_dataset = RawVideoDataset(CSV_PATH, split="train", resize_shape=DOWNSAMPLE_TO)
    
    # 2. Create data loader
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # 3. Number of classes
    num_classes = len(train_dataset.label_to_idx)
    print(f"Number of training samples: {len(train_dataset)}")
    print(f"Number of classes: {num_classes}")
    
    # 4. Instantiate model, loss, optimizer
    model = VideoLSTM(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # 5. Training (no val/test sets)
    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0.0
        correct, total = 0, 0
        
        for frames_batch, labels_batch in train_loader:
            frames_batch = frames_batch.to(device)   # (batch, 120, input_dim)
            labels_batch = labels_batch.to(device)   # (batch,)
            
            optimizer.zero_grad()
            outputs = model(frames_batch)           # (batch, num_classes)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # For a rough training accuracy measure
            _, preds = torch.max(outputs, dim=1)
            correct += (preds == labels_batch).sum().item()
            total += labels_batch.size(0)
        
        avg_train_loss = total_loss / len(train_loader) if len(train_loader) > 0 else 0
        train_acc = correct / total if total > 0 else 0
        
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Train Acc: {train_acc*100:.2f}%")
    
    # 6. Save the trained model
    torch.save(model.state_dict(), "lstm_model_no_val_test.pth")
    print("Model saved.")

    return model  # return the trained model if you want to do predictions later

###############################################################################
#                                  MAIN                                       #
###############################################################################
if __name__ == "__main__":
    trained_model = train_only_lstm()



Number of training samples: 20
Number of classes: 20
Epoch [1/5] | Train Loss: 3.0339 | Train Acc: 0.00%
Epoch [2/5] | Train Loss: 2.8955 | Train Acc: 10.00%
Epoch [3/5] | Train Loss: 2.7036 | Train Acc: 10.00%
Epoch [4/5] | Train Loss: 2.4736 | Train Acc: 10.00%
Epoch [5/5] | Train Loss: 2.2468 | Train Acc: 25.00%
Model saved.


In [2]:
import cv2
import torch
import numpy as np
import torch.nn.functional as F  # for softmax

def predict_video_with_distribution(
    video_npy_path,
    model,
    label_map,
    resize_shape=(64,64),
    device='cpu'
):
    """
    video_npy_path : str - path to the .npy file of shape (T, H, W, 3)
    model          : your trained LSTM model (in eval mode)
    label_map      : dict, mapping {class_idx: class_name}
    resize_shape   : (width, height) to match your training downsampling
    device         : 'cpu' or 'cuda'

    Returns:
      pred_label  : str, the label with highest probability
      prob_dict   : dict {label_name -> probability}, for all classes
    """
    # 1. Load frames (T, H, W, 3)
    frames = np.load(video_npy_path)  
    
    # 2. Downsample + flatten each frame
    processed_frames = []
    for frame in frames:
        # Resize
        small_frame = cv2.resize(frame, resize_shape)
        # Flatten to 1D
        flat_frame = small_frame.reshape(-1)  # shape (64*64*3)
        processed_frames.append(flat_frame)
    
    # 3. Convert to numpy -> torch
    processed_frames = np.array(processed_frames, dtype=np.float32)  
    # Insert batch dimension: (1, T, input_dim)
    processed_frames_tensor = torch.from_numpy(processed_frames).unsqueeze(0).to(device)
    
    # 4. Inference
    model.eval()
    with torch.no_grad():
        outputs = model(processed_frames_tensor)  # shape (1, num_classes)
    
    # 5. Convert logits to probabilities
    probs = F.softmax(outputs, dim=1)  # shape (1, num_classes)
    
    # 6. Get predicted class
    _, pred_idx = torch.max(probs, dim=1)  # shape (1,)
    pred_idx = pred_idx.item()  
    pred_label = label_map[pred_idx]  
    
    # 7. Build prob_dict: {label_name: probability}
    prob_array = probs.squeeze(0).cpu().numpy()  # shape (num_classes,)
    prob_dict = {}
    for i, p in enumerate(prob_array):
        class_name = label_map[i]
        prob_dict[class_name] = float(p)
    
    return pred_label, prob_dict


In [6]:
# Load the model
model_loaded = VideoLSTM(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_classes=20)
model_loaded.load_state_dict(torch.load("lstm_model_one_video_per_class.pth"))
model_loaded.to(device)
model_loaded.eval()

# Rebuild your idx_to_label mapping from the dataset
train_dataset = RawVideoDataset("one_video_per_class.csv", split="train", resize_shape=(64,64))
idx_to_label = {}
for label_str, idx_int in train_dataset.label_to_idx.items():
    idx_to_label[idx_int] = label_str

#test_video_path = "one_video_per_class/teach-1.npy"  # Example from your CSV
#test_video_path = "processed_frames_test/class/class-test.npy"  # Replace with your test file path
#test_video_path = "processed_frames_test/room/room-test2.npy"  # for example
test_video_path = "processed_frames_test/teach/teach-test.npy"  # for example

pred_label, prob_dist = predict_video_with_distribution(
    video_npy_path=test_video_path,
    model=trained_model,       # or model_loaded if you loaded from disk
    label_map=idx_to_label,
    resize_shape=(64,64),      # same as your training
    device=device
)

print("Given Label: Teach")
print("Predicted label:", pred_label)
print("Probabilities:")
for lbl, p in prob_dist.items():
    print(f"{lbl}: {p*100:.2f}%")


Given Label: Teach
Predicted label: teach
Probabilities:
answer: 6.68%
bicycle: 10.47%
book: 1.94%
break: 3.07%
car: 9.06%
class: 0.43%
correct: 1.62%
die: 2.47%
exam: 0.60%
how: 5.35%
left: 2.13%
lose: 0.75%
model: 7.75%
now: 10.85%
page: 11.40%
right: 2.20%
room: 0.42%
teach: 15.45%
train: 0.96%
walk: 6.40%


/var/folders/js/b36xj6rs3k1c5z78dsb4p3p00000gn/T/ipykernel_43518/3784445656.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_loaded.load_state_dict(torch.load("lstm